###Transform Circuits Data
1. Read bronze circuits table
2. Keep only the columns required for analytics (drop url column)
3. Standardise colmn names using snake case
4. Rename columns to make then more meaningful
5. Filter out rows where circuit_id is null
6. Remove duplicate records
7. Transform values of columns circuit_id and locality to Title Case
8. Write the transformed data to silver circuits table

Below changes are required to implement incremental load

1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
3. Add created_timestamp, updated_timestamp and batch_id to the silver table.
4. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of inserting/creating the record. It should not be updated during the merge.
    - Ensure the we are not overwriting the data in silver table by older bronze data (re-run scenario)

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/03.silver_helpers

In [0]:
from pyspark.sql import functions as f

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


In [0]:
#circuits_df = spark.read.option('versionAsOf', 0).table(bronze_table)

In [0]:
circuits_df = (
    spark.table(bronze_table).filter((f.col("batch_id") == v_batch_id))
) 

In [0]:
#display(circuits_df)

In [0]:
#circuits_select_df = circuits_df.select(
#    "circuitId",
#    "circuitName",
#    "lat",
#    "lng",
#    "locality",
#    "country",
#    "current_timestamp",
#    "source_file"
#)

In [0]:
circuits_select_df = circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("lng"),
    f.col("locality"),
    f.col("country"),
    f.col("current_timestamp"),
    f.col("source_file"),
    f.col("batch_id")
)

In [0]:
#display(circuits_select_df)

In [0]:
#circuits_renamed_df = (
#    circuits_select_df
#    .withColumnRenamed("circuitId", "circuit_id")
#    .withColumnRenamed("circuitName", "circuit_name")
#    .withColumnRenamed("lat", "latitude")
#    .withColumnRenamed("lng", "longitude")
#)

In [0]:
circuits_renamed_df = (
    circuits_select_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "circuitName": "circuit_name",
        "lat": "latitude",
        "lng": "longitude"
        })
)

In [0]:
#display(circuits_renamed_df)

In [0]:
#circuits_valid_df = (
#    circuits_renamed_df.filter(
#       "circuit_id IS NOT NULL"
#    )
#)

In [0]:
circuits_valid_df = (
    circuits_renamed_df.filter(
       f.col("circuit_id").isNotNull()
    )
)

In [0]:
#display(circuits_valid_df)

In [0]:
#circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
#display(circuits_distinct_df)

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name',f.initcap(f.col("circuit_name")))
        .withColumn('locality', f.initcap(f.col("locality")))

)

In [0]:
write_to_silver(
    circuits_final_df,
    silver_table,
    "t.circuit_id = s.circuit_id",
    [
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "current_timestamp",
        "source_file",
        "batch_id"
    ]

)

In [0]:
#display(circuits_final_df)

In [0]:
#if not spark.catalog.tableExists(silver_table):
#    (
#        circuits_final_df
#        .write
#        .mode("overwrite")
#        .format("delta")
#        .saveAsTable(silver_table)
#    )
#else:
#    from delta.tables import DeltaTable
#    
#    delta_table = DeltaTable.forName(spark, silver_table)
#    (
#        delta_table.alias("t")
#        .merge(
#            circuits_final_df.alias("s"),
#            "t.circuit_id = s.circuit_id"
#        )
#        .whenMatchedUpdate(
#            condition = "s.batch_id >= t.batch_id",
#            set = {
#                "circuit_name": "s.circuit_name",
#                "latitude": "s.latitude",
#                "longitude": "s.longitude",
#                "locality": "s.locality",
#                "country": "s.country",
#                "current_timestamp": "s.current_timestamp",
#                "source_file": "s.source_file",
#                "batch_id": "s.batch_id",
#                "updated_timestamp": "s.updated_timestamp"
#            }
#            
#        )
#        .whenNotMatchedInsertAll()
#        .execute()
#    )

In [0]:
#display(spark.table(silver_table))

In [0]:
%sql
--select * from formula1.silver.circuits